# Issam — One Video Sample Test

Classify the **bundled** KARSL clip with no webcam and no external dataset.

| Field | Value |
|-------|-------|
| Video | `sample_videos/skeleton_SignID0071.mp4` |
| Expected | **Skeleton** / هيكل عظمي (SignID **71**, class index **0**) |
| Model | issam 100-word BiLSTM (225-D Holistic, 48 frames) |

**Run:** Cell 1 → 2 → 3

For live webcam + checklist use `Issam_Word_Live_Test.ipynb` in this same folder.

In [ ]:
# CELL 1 — imports
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import tensorflow as tf
from pathlib import Path

os.environ.setdefault('PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION', 'python')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

print(f'TF {tf.__version__}  |  MP {mp.__version__}  |  CV2 {cv2.__version__}')

In [ ]:
# CELL 2 — bundle paths + model + labels
from bundle_config import find_bundle_from_cwd, MODEL_PATH, LABELS_XLSX, SAMPLE_VIDEO, F_AVG, N_FEATURES

DIR = find_bundle_from_cwd()
print(f'Bundle root: {DIR}')
EXPECTED_CLASS_INDEX = 0  # SignID 71 = Skeleton

karsl_df = pd.read_excel(LABELS_XLSX)
karsl_100 = karsl_df.iloc[70:170].reset_index(drop=True)
words_en = karsl_100['Sign-English'].astype(str).str.strip().to_numpy()
words_ar = karsl_100['Sign-Arabic'].astype(str).str.strip().to_numpy()
sign_ids = karsl_100['SignID'].astype(int).to_numpy()
num_classes = len(words_ar)

def build_model(n_classes):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(F_AVG, N_FEATURES)),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(n_classes, activation='softmax'),
    ])

try:
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    print('[model] loaded full')
except Exception as e:
    print(f'[model] rebuild + weights ({e})')
    model = build_model(num_classes)
    model.load_weights(MODEL_PATH)

assert SAMPLE_VIDEO.exists(), f'Missing bundled video: {SAMPLE_VIDEO}'
print(f'Bundle : {DIR}')
print(f'Video  : {SAMPLE_VIDEO.name}')
print(f'Target : [{EXPECTED_CLASS_INDEX}] {words_ar[EXPECTED_CLASS_INDEX]} / {words_en[EXPECTED_CLASS_INDEX]}  SignID {sign_ids[EXPECTED_CLASS_INDEX]}')

In [ ]:
# CELL 3 — extract + classify bundled sample video
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

def mediapipe_detection(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False
    return holistic.process(rgb)

def adjust_landmarks(arr, center):
    arr_reshaped = arr.reshape(-1, 3)
    return (arr_reshaped - np.tile(center, (len(arr_reshaped), 1))).reshape(-1)

def extract_keypoints(results):
    pose = np.array([[r.x, r.y, r.z] for r in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(99)
    lh = np.array([[r.x, r.y, r.z] for r in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(63)
    rh = np.array([[r.x, r.y, r.z] for r in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(63)
    return np.concatenate([
        adjust_landmarks(pose, pose[:3]),
        adjust_landmarks(lh, lh[:3]),
        adjust_landmarks(rh, rh[:3]),
    ]).astype(np.float32)

def pad_or_trim(seq, f_avg=F_AVG):
    seq = np.asarray(seq, dtype=np.float32)
    n = min(seq.shape[0], f_avg)
    seq = seq[:n, :]
    while seq.shape[0] < f_avg:
        seq = np.concatenate([seq, seq[-1:, :]], axis=0)
    return seq

def predict_video(video_path, expected_index=None, topk=5):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(video_path)
    seq, n_frames = [], 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        seq.append(extract_keypoints(mediapipe_detection(frame)))
        n_frames += 1
    cap.release()
    probs = model.predict(pad_or_trim(seq)[None, ...], verbose=0)[0]
    order = np.argsort(probs)[::-1][:topk]
    print(f'\n=== {Path(video_path).name} ({n_frames} frames) ===')
    for rank, i in enumerate(order, 1):
        mark = '  <-- expected' if expected_index is not None and i == expected_index else ''
        print(f'  {rank}. [{i:2d}] SignID {sign_ids[i]}  {words_ar[i]}  /  {words_en[i]}  ({probs[i]*100:.1f}%){mark}')
    top = int(order[0])
    if expected_index is not None:
        ok = top == expected_index
        print(f'  Result: {"CORRECT" if ok else "WRONG"} (top={words_en[top]})')
    return top, float(probs[top]), n_frames

top_i, top_conf, frames = predict_video(SAMPLE_VIDEO, expected_index=EXPECTED_CLASS_INDEX)
holistic.close()
print(f'\nDone — {frames} frames processed, top confidence {top_conf:.1%}')